# Load data and train/test split

Everyone uses this. Run these cells first.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)
import matplotlib.pyplot as plt

df = pd.read_parquet("combined_preprocessed.parquet")
print(df.shape)


(6287183, 122)


In [ ]:
X = df.drop(columns=["ARR_DEL15"])
y = df["ARR_DEL15"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Delay rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")


Train: (5029746, 121), Test: (1257437, 121)
Delay rate — train: 0.226, test: 0.226


Helper function so everyone reports the same metrics.

In [ ]:
def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1:        {f1_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["On Time", "Delayed"]))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    axes[0].imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=14)
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(["On Time", "Delayed"])
    axes[0].set_yticklabels(["On Time", "Delayed"])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title(f"{name} — Confusion Matrix")

    # ROC curve
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], name=name)
    axes[1].set_title(f"{name} — ROC Curve")

    plt.tight_layout()
    plt.show()

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "AUC-ROC": roc_auc_score(y_test, y_prob),
    }


In [4]:
results = []

---
# Wahid — Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

# TODO (Wahid):
# 1. Create the model
# 2. Fit on X_train, y_train
# 3. results.append(evaluate('Naive Bayes', nb_model, X_test, y_test))


---
# Sam — Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO (Sam):
# 1. Create model with: max_iter=1000, class_weight='balanced', random_state=42
# 2. Fit on X_train, y_train
# 3. results.append(evaluate('Logistic Regression', lr_model, X_test, y_test))


In [ ]:
# TODO (Sam): Feature importance plot
# 1. Get coefficients from lr_model.coef_[0]
# 2. Find the top 15 by absolute value
# 3. Plot as horizontal bar chart


---
# Aryan — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO (Aryan):
# 1. Create model with: n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
# 2. Fit on X_train, y_train
# 3. results.append(evaluate('Random Forest', rf_model, X_test, y_test))


In [ ]:
# TODO (Aryan): Feature importance plot
# 1. Get importances from rf_model.feature_importances_
# 2. Find the top 15
# 3. Plot as horizontal bar chart


---
# Dennis — RNN

Needs `pip install tensorflow`.

In [ ]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

# TODO (Dennis): Preprocessing
# 1. Scale X_train and X_test with StandardScaler
# 2. Reshape to 3D: (samples, 1, features) — RNN expects 3D input


In [ ]:
# TODO (Dennis): Build and train
# 1. Build Sequential: SimpleRNN(64) -> Dropout(0.3) -> Dense(32) -> Dense(1, sigmoid)
# 2. Compile with optimizer='adam', loss='binary_crossentropy'
# 3. Fit with epochs=10, batch_size=1024, validation_split=0.1


In [ ]:
# TODO (Dennis): Evaluate
# 1. Get probabilities: y_prob = rnn_model.predict(X_test_rnn).flatten()
# 2. Get predictions: y_pred = (y_prob >= 0.5).astype(int)
# 3. Print metrics (accuracy, precision, recall, F1, AUC-ROC)
# 4. Plot confusion matrix and ROC curve
# 5. Append results dict to results list


---
# Compare all models

In [ ]:
comparison = pd.DataFrame(results).set_index('Model')
comparison.round(4)

In [ ]:
comparison.plot.bar(figsize=(10, 5), title='Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()